# 📊 **YOLO People Detection & Counting System**

## 🎯 Objective
This project builds a computer vision system to detect and count people using YOLO (You Only Look Once). The system is designed for real-world applications such as foot traffic analysis in offices, malls, and public spaces.

## 🚀 Key Features
- Real-time person detection using YOLOv8
- Model fine-tuning on custom dataset
- Evaluation using validation metrics
- Inference on images and videos
- Foundation for people counting system

## 🛠️ Technologies
- Python
- OpenCV
- Ultralytics YOLOv8

# **IMPORTING LIBRARIES**

This section loads all required libraries for data handling, visualization, and model training.

In [ ]:
# Import Libraries

import os
import cv2
import yaml
import csv
import random
import shutil
import logging
import matplotlib.pyplot as plt

from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO

import warnings
warnings.filterwarnings('ignore')

# **LOADING AND VIEWING DATASET**

We inspect the dataset structure to ensure images and labels are correctly organized for YOLO training.

## Step 1: Loading The Dataset

In [12]:
DATASET_PATH = r"C:\Users\BEST\Desktop\2026_projects\YOLO_People_Counting_System\data"

for root, dirs, files in os.walk(DATASET_PATH):
    print(f"{root} -> {len(files)} files")

C:\Users\BEST\Desktop\2026_projects\YOLO_People_Counting_System\data -> 1 files
C:\Users\BEST\Desktop\2026_projects\YOLO_People_Counting_System\data\train -> 15211 files
C:\Users\BEST\Desktop\2026_projects\YOLO_People_Counting_System\data\valid -> 0 files
C:\Users\BEST\Desktop\2026_projects\YOLO_People_Counting_System\data\valid\valid -> 1432 files


In [ ]:
# Verfying the directory path exists.

BASE_DIR = Path.cwd().parent

DATA_PATH = BASE_DIR / "data"

print("Base dir:", BASE_DIR)
print("Data path exists:", DATA_PATH.exists())

Base dir: c:\Users\BEST\Desktop\2026_projects\YOLO_People_Counting_System
Data path exists: True


In [ ]:
base = DATA_PATH / "valid"

subfolders = [p for p in base.iterdir() if p.is_dir()]
print("Subfolders found:", subfolders)

if len(subfolders) == 1:
    src = subfolders[0]
    dst = base

    for file in src.iterdir():
        shutil.move(str(file), str(dst))

    src.rmdir()
    print("✅ Structure fixed!")

Subfolders found: [WindowsPath('c:/Users/BEST/Desktop/2026_projects/YOLO_People_Counting_System/data/valid/valid')]
✅ Structure fixed!


In [21]:
print("Train exists:", (DATA_PATH / "train").exists())
print("Valid exists:", (DATA_PATH / "valid").exists())

Train exists: True
Valid exists: True


In [22]:
DATASET_PATH = r"C:\Users\BEST\Desktop\2026_projects\YOLO_People_Counting_System\data"

for root, dirs, files in os.walk(DATASET_PATH):
    print(f"{root} -> {len(files)} files")

C:\Users\BEST\Desktop\2026_projects\YOLO_People_Counting_System\data -> 1 files
C:\Users\BEST\Desktop\2026_projects\YOLO_People_Counting_System\data\train -> 15211 files
C:\Users\BEST\Desktop\2026_projects\YOLO_People_Counting_System\data\valid -> 1432 files


## Step 2: Dataset Conversion (CSV-YOLO)

Convert annotations from CSV format into YOLO label format.

In [26]:
ls C:\Users\BEST\Desktop\2026_projects\YOLO_People_Counting_System\data\train\

 Volume in drive C has no label.
 Volume Serial Number is ECC5-0379

 Directory of C:\Users\BEST\Desktop\2026_projects\YOLO_People_Counting_System\data\train

04/10/2026  04:07 PM    <DIR>          .
04/11/2026  10:06 PM    <DIR>          ..
04/10/2026  03:56 PM         8,812,523 _annotations.csv
04/10/2026  03:46 PM           257,282 000001_jpg.rf.896508b6d1874829ee5d32d736158d29.jpg
04/10/2026  03:46 PM           302,663 000001_jpg.rf.c82cb40fa9d0494897fb9ce8157a3dc8.jpg
04/10/2026  03:46 PM           184,557 000001_jpg.rf.fd11b57224ecddaec5af674c447c67f4.jpg
04/10/2026  03:46 PM           215,190 000003_jpg.rf.27f5678c703b1852a2779889278e70d5.jpg
04/10/2026  03:46 PM           188,508 000003_jpg.rf.7ec28422125b41630b7a6a5599793ac4.jpg
04/10/2026  03:46 PM           250,118 000003_jpg.rf.a1715bd43b0534b15e1aea18d2e2fa19.jpg
04/10/2026  03:46 PM           344,207 000007_jpg.rf.04684d55fa4b8a8d7583bb4377206486.jpg
04/10/2026  03:46 PM           184,685 000007_jpg.rf.168e1d1bca0a8ed616f

In [27]:
ls C:\Users\BEST\Desktop\2026_projects\YOLO_People_Counting_System\data\train\

 Volume in drive C has no label.
 Volume Serial Number is ECC5-0379

 Directory of C:\Users\BEST\Desktop\2026_projects\YOLO_People_Counting_System\data\train

04/10/2026  04:07 PM    <DIR>          .
04/11/2026  10:06 PM    <DIR>          ..
04/10/2026  03:56 PM         8,812,523 _annotations.csv
04/10/2026  03:46 PM           257,282 000001_jpg.rf.896508b6d1874829ee5d32d736158d29.jpg
04/10/2026  03:46 PM           302,663 000001_jpg.rf.c82cb40fa9d0494897fb9ce8157a3dc8.jpg
04/10/2026  03:46 PM           184,557 000001_jpg.rf.fd11b57224ecddaec5af674c447c67f4.jpg
04/10/2026  03:46 PM           215,190 000003_jpg.rf.27f5678c703b1852a2779889278e70d5.jpg
04/10/2026  03:46 PM           188,508 000003_jpg.rf.7ec28422125b41630b7a6a5599793ac4.jpg
04/10/2026  03:46 PM           250,118 000003_jpg.rf.a1715bd43b0534b15e1aea18d2e2fa19.jpg
04/10/2026  03:46 PM           344,207 000007_jpg.rf.04684d55fa4b8a8d7583bb4377206486.jpg
04/10/2026  03:46 PM           184,685 000007_jpg.rf.168e1d1bca0a8ed616f

In [ ]:
CLASS_MAP = {"person": 0}

def bbox_to_yolo(xmin, ymin, xmax, ymax, width, height):
    x_center = ((xmin + xmax) / 2.0) / width
    y_center = ((ymin + ymax) / 2.0) / height
    w = (xmax - xmin) / width
    h = (ymax - ymin) / height
    return [x_center, y_center, w, h]


def convert_annotations(csv_path, images_dir, labels_dir):
    labels_dir.mkdir(parents=True, exist_ok=True)
    rows_by_image = defaultdict(list)

    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            if row["class"] not in CLASS_MAP:
                continue

            filename = row["filename"]
            width = float(row["width"])
            height = float(row["height"])

            xmin = float(row["xmin"])
            ymin = float(row["ymin"])
            xmax = float(row["xmax"])
            ymax = float(row["ymax"])

            yolo_box = bbox_to_yolo(xmin, ymin, xmax, ymax, width, height)
            class_id = CLASS_MAP[row["class"]]

            rows_by_image[filename].append(
                f"{class_id} {' '.join(map(str, yolo_box))}"
            )

    for filename, lines in rows_by_image.items():
        label_path = labels_dir / (Path(filename).stem + ".txt")
        label_path.write_text("\n".join(lines))

## Step 3: Run Coversion

In [ ]:
repo_root = Path.cwd()

convert_annotations(
    repo_root / "data/train/_annotations.csv",
    repo_root / "data/train",
    repo_root / "data/train/labels"
)

convert_annotations(
    repo_root / "data/valid/_annotations.csv",
    repo_root / "data/valid",
    repo_root / "data/valid/labels"
)

# **EXPLORATORY DATA ANALYSIS**

We visualize sample images to understand the dataset distribution and quality.

In [ ]:
# ============================================
# 🖼️ Visualize Sample Images
# ============================================

image_dir = os.path.join(DATASET_PATH, "images/train")

images = os.listdir(image_dir)

sample_images = random.sample(images, 5)

plt.figure(figsize=(15, 8))

for i, img_name in enumerate(sample_images):
    img_path = os.path.join(image_dir, img_name)
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(1, 5, i+1)
    plt.imshow(img)
    plt.title(img_name)
    plt.axis("off")

plt.show()

In [ ]:
# ============================================
# 🧾 Create data.yaml
# ============================================

data_config = {
    "train": "dataset/train",
    "val": "dataset/val",
    "nc": 1,
    "names": ["person"]
}

with open("data.yaml", "w") as f:
    yaml.dump(data_config, f)

print("data.yaml created!")

# **TRAINING MODEL**

## Step 1: Loading YOLO Model

We load a pre-trained YOLOv8 model to fine-tune on our dataset.

In [ ]:
# ============================================
# 🧠 Load YOLO Model
# ============================================

model = YOLO("yolov8n.pt")  # pretrained

print("Model loaded successfully!")

## Step 2: Hyperparameter Tuning

We experiment with different hyperparameters to optimize model performance.

In [ ]:
model.tune(
    data="data.yaml",
    epochs=30,
    iterations=20,
    optimizer="Adam"
)

# **MODEL SAVING**

Saving trained model weights and metadata for future tracking and deployment.

In [ ]:
from datetime import datetime

run_name = "people_model_" + datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = Path("saved_models") / run_name
save_dir.mkdir(parents=True, exist_ok=True)

# Path to YOLO trained weights
best_model_path = Path("runs/detect/tuned_people_model/weights/best.pt")

# Copy weights
shutil.copy(best_model_path, save_dir / "model.pt")

# Save metadata
metadata = {
    "model": "YOLOv8",
    "epochs": 50,
    "batch_size": 16,
    "img_size": 640,
    "learning_rate": 0.001
}

with open(save_dir / "metadata.yaml", "w") as f:
    yaml.dump(metadata, f)

print(f"Model saved at: {save_dir}")

# **MODEL EVALUATION**

We evaluate the trained model using validation data to measure performance.

In [ ]:
# ============================================
# 📊 Model Evaluation
# ============================================

eval_model = YOLO(best_model_path)
metrics = eval_model.val(data="data.yaml")

print(metrics)

# **INFERENCE ON IMAGE AND VIDEO (People Counting Base)**

We test the trained model on unseen images.

In [ ]:
results = eval_model.predict("test.jpg", conf=0.5)
results[0].show()

In [ ]:
# ============================================
# 🔍 Inference on Image
# ============================================

test_image = "dataset/images/val/sample.jpg"

results = model.predict(test_image, conf=0.5)

results[0].show()

We apply the model to video data for real-world testing.

In [ ]:
# ============================================
# 🎥 Video Inference
# ============================================

video_path = "video.mp4"

cap = cv2.VideoCapture(video_path)

model = YOLO("runs/detect/yolo_people_detector/weights/best.pt")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)

    annotated_frame = results[0].plot()

    cv2.imshow("YOLO Detection", annotated_frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()